# 03 - Transfer Learning Softmax CNN

**Purpose:** a pretrained CNN learns visual features automatically, while the
softmax output guarantees a valid composition vector.

We freeze a pretrained EfficientNet-B0 backbone (too little data to fine-tune
5M weights on 161 mixtures) and train only a small softmax head. The question
is whether learned CNN features beat the hand-crafted Random Forest baseline.


## 1. Imports and setup


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src import config, data, targets, splits, metrics, plots, models, training

config.ensure_output_dirs()
training.set_seed(config.RANDOM_SEED)
device = training.get_device()
print("device:", device)


## 2-5. Load metadata, build targets, validate


In [ ]:
metadata = data.load_metadata()
image_df = data.build_image_dataframe(metadata)

vocabulary = targets.build_material_vocabulary(metadata)
target_vectors = targets.build_targets(metadata, vocabulary=vocabulary)
materials = target_vectors["materials"]

report = data.validate_dataset(metadata)
assert report["ok"], report["problems"]
targets.validate_targets(target_vectors)
print(f"{len(metadata)} mixtures, {len(image_df)} images, materials = {materials}")


## 6. Grouped train / validation / test split by Code

As in every notebook, all five photos of a mixture stay in one split so the
model is never tested on a mixture it has already seen.


In [ ]:
train_df, val_df, test_df = splits.grouped_train_val_test_split(
    image_df, group_col="Code",
    test_size=config.TEST_SIZE, val_size=config.VAL_SIZE, seed=config.RANDOM_SEED,
)
splits.check_no_leakage(train_df, val_df, test_df)
splits.check_groups_intact([train_df, val_df, test_df], image_df)
print("images -> train {}, val {}, test {}".format(len(train_df), len(val_df), len(test_df)))
print("codes  -> train {}, val {}, test {}".format(
    train_df["Code"].nunique(), val_df["Code"].nunique(), test_df["Code"].nunique()))


## 7. Training target: volume fraction

The image model is trained on **volume fractions** because photographs reflect
visible area/volume and texture more directly than mass. Predictions are also
converted back to **mass space** for comparison with the Random Forest baseline.


In [ ]:
volume_by_code = {c: target_vectors["volume_fraction"][i]
                  for i, c in enumerate(target_vectors["codes"])}
mass_by_code = {c: target_vectors["mass_fraction"][i]
                for i, c in enumerate(target_vectors["codes"])}
density_vector = target_vectors["density_matrix"][0]   # global per-material densities


## 8. Datasets and dataloaders

Each photo is decoded once into a small in-memory image (the backbone is frozen,
so its input never needs re-decoding); mild augmentation is then applied on the
fly during training.


In [ ]:
image_cache = training.preload_images(image_df, config.IMAGE_SIZE)

train_tf = training.build_transforms(train=True, image_size=config.IMAGE_SIZE)
eval_tf = training.build_transforms(train=False, image_size=config.IMAGE_SIZE)

def make_dataset(df, tf):
    return training.CompositionImageDataset(
        df, volume_by_code, tf, image_size=config.IMAGE_SIZE, image_cache=image_cache)

train_ds = make_dataset(train_df, train_tf)        # augmented, for training
train_eval_ds = make_dataset(train_df, eval_tf)    # deterministic, for prediction
val_ds = make_dataset(val_df, eval_tf)
test_ds = make_dataset(test_df, eval_tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
train_eval_loader = DataLoader(train_eval_ds, batch_size=16, shuffle=False, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)
print("batches -> train", len(train_loader), "val", len(val_loader), "test", len(test_loader))


## 9. Build the model: frozen EfficientNet-B0 + softmax head


In [ ]:
backbone, embedding_dim = models.build_efficientnet_backbone(pretrained=True, freeze=True)
model = models.CompositionCNN(backbone, embedding_dim, len(materials), dropout=0.3)
print("embedding dim:", embedding_dim)
print("trainable parameters (head only):", models.count_trainable_parameters(model))


## 10. Train the softmax head

KL divergence is the natural loss (both prediction and target are
distributions); MAE is reported for readability. Early stopping watches the
validation loss.


In [ ]:
t0 = time.time()
model, history = training.train_composition_model(
    model, train_loader, val_loader, device,
    epochs=30, lr=1e-3, patience=7, loss="kl",
    save_path=config.MODEL_DIR / "composition_cnn_softmax.pt",
)
train_seconds = time.time() - t0
best = history.loc[history["val_loss"].idxmin()]
print(f"training took {train_seconds:.0f}s over {len(history)} epochs")
print(f"best epoch {int(best['epoch'])}: val_loss {best['val_loss']:.4f}, "
      f"val_mae {best['val_mae']:.4f}")


## 11. Training curves


In [ ]:
plots.plot_training_curve(history, save_path="03_training_curve.png")
plt.show()


## 12. Predict and verify valid composition vectors


In [ ]:
pred_train = training.predict_composition_model(model, train_eval_loader, device, materials, "train")
pred_val = training.predict_composition_model(model, val_loader, device, materials, "val")
pred_test = training.predict_composition_model(model, test_loader, device, materials, "test")

# Predictions are in VOLUME space (the training target). Verify they are valid.
pred_cols = [f"pred_{m}" for m in materials]
for df_ in (pred_train, pred_val, pred_test):
    P = df_[pred_cols].to_numpy()
    assert (P >= 0).all() and np.allclose(P.sum(axis=1), 1.0, atol=1e-5)
print("predictions are valid compositions (non-negative, rows sum to 1)")


## 13. Evaluate in volume space (image level)


In [ ]:
def vol_arrays(pred_df):
    yt = pred_df[[f"true_{m}" for m in materials]].to_numpy()
    yp = pred_df[[f"pred_{m}" for m in materials]].to_numpy()
    return yt, yp

vol_metrics = {name: metrics.evaluate_composition(*vol_arrays(df_), materials)
               for name, df_ in [("train", pred_train), ("val", pred_val), ("test", pred_test)]}
vol_df = pd.DataFrame(vol_metrics).T
vol_df.index.name = "split"
vol_df[["MAE", "RMSE", "R2", "presence_F1", "false_positive_absent_%"]].round(4)


## 14. Convert to mass space and evaluate

`mass_i = volume_i * density_i`, renormalized to sum to 1. We then evaluate in
mass space so the CNN is directly comparable with the Random Forest baseline.


In [ ]:
mass_metrics, mass_pred = {}, {}
for name, df_ in [("train", pred_train), ("val", pred_val), ("test", pred_test)]:
    yp_vol = df_[[f"pred_{m}" for m in materials]].to_numpy()
    yp_mass = targets.volume_to_mass(yp_vol, density_vector)
    yt_mass = np.array([mass_by_code[c] for c in df_["Code"]])
    mass_metrics[name] = metrics.evaluate_composition(yt_mass, yp_mass, materials)
    mass_pred[name] = (yt_mass, yp_mass)

mass_df = pd.DataFrame(mass_metrics).T
mass_df.index.name = "split"
mass_df[["MAE", "RMSE", "R2", "presence_F1", "false_positive_absent_%"]].round(4)


## 15. Mixture-level evaluation

Average the five image predictions per Code, renormalize, and score against the
Code-level target -- the physical mixture is the real object of interest.


In [ ]:
def mixture_level(pred_df, y_pred_image, truth_by_code):
    tmp = pd.DataFrame(y_pred_image, columns=materials)
    tmp["Code"] = pred_df["Code"].values
    grouped = tmp.groupby("Code")[materials].mean()
    codes = grouped.index.tolist()
    pred_mix = training.renormalize_composition(grouped.to_numpy())
    y_true = np.array([truth_by_code[c] for c in codes])
    return codes, y_true, pred_mix

# mass space (for RF comparison)
yt_mass_test, yp_mass_test = mass_pred["test"]
test_codes_mix, y_test_mix_mass, pred_test_mix_mass = mixture_level(
    pred_test, yp_mass_test, mass_by_code)
mix_mass_metrics = metrics.evaluate_composition(y_test_mix_mass, pred_test_mix_mass, materials)

# volume space (native)
yp_vol_test = pred_test[[f"pred_{m}" for m in materials]].to_numpy()
_, y_test_mix_vol, pred_test_mix_vol = mixture_level(pred_test, yp_vol_test, volume_by_code)
mix_vol_metrics = metrics.evaluate_composition(y_test_mix_vol, pred_test_mix_vol, materials)

show = ["MAE", "RMSE", "R2", "presence_F1", "false_positive_absent_%"]
print("test mixture-level (mass)  :", {k: round(mix_mass_metrics[k], 4) for k in show})
print("test mixture-level (volume):", {k: round(mix_vol_metrics[k], 4) for k in show})


## 16. Compare with the Random Forest baseline

Same metric functions, same grouped split. The mass-space rows are the fair
head-to-head comparison.


In [ ]:
rf = pd.read_csv(config.TABLE_DIR / "baseline_random_forest_metrics.csv", index_col="split")
cols = ["MAE", "RMSE", "R2", "presence_F1", "false_positive_absent_%"]

comparison_df = pd.DataFrame([
    {"Model": "Random Forest", "Level": "image",   "Space": "mass",   **rf.loc["test", cols].to_dict()},
    {"Model": "Random Forest", "Level": "mixture", "Space": "mass",   **rf.loc["test_mixture", cols].to_dict()},
    {"Model": "Softmax CNN",   "Level": "image",   "Space": "volume", **{k: vol_metrics["test"][k] for k in cols}},
    {"Model": "Softmax CNN",   "Level": "image",   "Space": "mass",   **{k: mass_metrics["test"][k] for k in cols}},
    {"Model": "Softmax CNN",   "Level": "mixture", "Space": "volume", **{k: mix_vol_metrics[k] for k in cols}},
    {"Model": "Softmax CNN",   "Level": "mixture", "Space": "mass",   **{k: mix_mass_metrics[k] for k in cols}},
])[["Model", "Level", "Space"] + cols].round(4)
comparison_df


## 17. Save metrics and predictions


In [ ]:
rows = []
for name in ["train", "val", "test"]:
    rows.append({"split": name, "level": "image", "space": "volume", **vol_metrics[name]})
for name in ["train", "val", "test"]:
    rows.append({"split": name, "level": "image", "space": "mass", **mass_metrics[name]})
rows.append({"split": "test", "level": "mixture", "space": "mass", **mix_mass_metrics})
rows.append({"split": "test", "level": "mixture", "space": "volume", **mix_vol_metrics})
metrics_df = pd.DataFrame(rows)
metrics_path = config.TABLE_DIR / "transfer_learning_softmax_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

# Image-level predictions: mass columns (comparable to RF) + native volume columns.
def predictions_frame(pred_df):
    yp_vol = pred_df[[f"pred_{m}" for m in materials]].to_numpy()
    yt_vol = pred_df[[f"true_{m}" for m in materials]].to_numpy()
    yp_mass = targets.volume_to_mass(yp_vol, density_vector)
    yt_mass = np.array([mass_by_code[c] for c in pred_df["Code"]])
    out = pd.DataFrame({"Code": pred_df["Code"].values,
                        "image_path": pred_df["image_path"].values,
                        "split": pred_df["split"].values})
    for j, m in enumerate(materials):
        out[f"true_{m}"] = yt_mass[:, j]
        out[f"pred_{m}"] = yp_mass[:, j]
        out[f"true_vol_{m}"] = yt_vol[:, j]
        out[f"pred_vol_{m}"] = yp_vol[:, j]
    return out

pred_all = pd.concat([predictions_frame(pred_train), predictions_frame(pred_val),
                      predictions_frame(pred_test)], ignore_index=True)
pred_path = config.PREDICTION_DIR / "transfer_learning_softmax_predictions.csv"
pred_all.to_csv(pred_path, index=False)
print("saved:", metrics_path.name, "|", pred_path.name, "| composition_cnn_softmax.pt")


## 18. Plots


In [ ]:
yt_mass_test, yp_mass_test = mass_pred["test"]
plots.plot_predicted_vs_true(yt_mass_test, yp_mass_test, materials,
                             save_path="03_pred_vs_true_test.png")
plt.show()

plots.plot_per_material_mae(metrics.per_material_mae(yt_mass_test, yp_mass_test, materials),
                            save_path="03_per_material_mae_test.png")
plt.show()

example_codes = test_codes_mix[:4]
plots.plot_example_predictions(image_df, example_codes, y_test_mix_mass[:4],
                               pred_test_mix_mass[:4], materials,
                               save_path="03_example_predictions_test.png")
plt.show()


## 19. Interpretation

**Does the CNN beat the Random Forest?** Compare the mass-space, mixture-level
rows in the table above: that is the fair head-to-head against the baseline
(RF mixture-level MAE 0.125 / R^2 0.41). Learned features should match or beat
the hand-crafted ones.

**Do pretrained features help?** The backbone is frozen ImageNet EfficientNet-B0
and only a ~7.7k-parameter head is trained, yet it learns a useful mapping from
generic visual features to composition -- evidence that pretrained features
transfer to this domain.

**Does absent-material leakage remain?** Softmax produces a valid simplex
(`composition_sum_error` ~ 0) but can never output an exact zero, so some
predicted mass still lands on absent materials (`false_positive_absent_%`). The
baseline had ~32%; softmax typically lowers this but does not eliminate it.

**Why this motivates Phase 4.** Each mixture has only 1-3 of the 6 materials, but
neither RF nor softmax can output hard zeros. A sparse multitask model with an
explicit presence head (Phase 4) is designed to drive the absent fractions to
zero and cut that leakage further.
